# Praktikum: Kredensial Mikro Terverifikasi Blockchain + Analitik Pembelajaran
Pendamping monograf *Kredensial Mikro Terverifikasi Blockchain* (judul kerja). Berjalan di Google Colab atau Jupyter lokal tanpa node blockchain eksternal (memakai **eth-tester**).

**Tujuan:** (1) mengompilasi dan menyebarkan smart contract registri kredensial; (2) menerbitkan, memverifikasi, dan mencabut kredensial berformat W3C VC 2.0/Open Badges 3.0; (3) mengukur gas; (4) melatih model prediksi capaian dini dengan OULAD.

In [1]:
import importlib, subprocess, sys
for pkg, mod in [('py-solc-x','solcx'),('web3','web3'),('eth-tester[py-evm]','eth_tester'),('scikit-learn','sklearn'),('pandas','pandas'),('matplotlib','matplotlib')]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('paket siap')

paket siap


## 1. Kompilasi smart contract

In [2]:
SOURCE = r'''// SPDX-License-Identifier: MIT
pragma solidity ^0.8.24;

/// @title CredentialRegistry — registri jangkar hash kredensial mikro (prototipe monograf)
/// @notice Menyimpan hanya hash (bytes32) kredensial; dokumen kredensial lengkap disimpan di luar rantai.
contract CredentialRegistry {
    struct Issuer { string name; bool active; uint64 registeredAt; }
    struct Record { address issuer; uint64 issuedAt; uint64 revokedAt; bytes32 subjectRef; }

    address public admin;
    mapping(address => Issuer) public issuers;
    mapping(bytes32 => Record) private records;
    mapping(bytes32 => address) public batchRoots; // akar Merkle -> penerbit

    event IssuerAdded(address indexed issuer, string name);
    event IssuerDeactivated(address indexed issuer);
    event CredentialIssued(bytes32 indexed credHash, address indexed issuer, bytes32 subjectRef);
    event CredentialRevoked(bytes32 indexed credHash, address indexed issuer, string reason);
    event BatchAnchored(bytes32 indexed root, address indexed issuer, uint256 size);

    error NotAdmin(); error NotActiveIssuer(); error AlreadyExists(); error UnknownCredential();
    error NotOriginalIssuer(); error AlreadyRevoked();

    modifier onlyAdmin() { if (msg.sender != admin) revert NotAdmin(); _; }
    modifier onlyIssuer() { if (!issuers[msg.sender].active) revert NotActiveIssuer(); _; }

    constructor() { admin = msg.sender; }

    function addIssuer(address issuer, string calldata name) external onlyAdmin {
        issuers[issuer] = Issuer(name, true, uint64(block.timestamp));
        emit IssuerAdded(issuer, name);
    }

    function deactivateIssuer(address issuer) external onlyAdmin {
        issuers[issuer].active = false;
        emit IssuerDeactivated(issuer);
    }

    function issue(bytes32 credHash, bytes32 subjectRef) external onlyIssuer {
        if (records[credHash].issuedAt != 0) revert AlreadyExists();
        records[credHash] = Record(msg.sender, uint64(block.timestamp), 0, subjectRef);
        emit CredentialIssued(credHash, msg.sender, subjectRef);
    }

    function issueBatch(bytes32[] calldata hashes, bytes32 subjectRef) external onlyIssuer {
        for (uint256 i = 0; i < hashes.length; i++) {
            if (records[hashes[i]].issuedAt != 0) revert AlreadyExists();
            records[hashes[i]] = Record(msg.sender, uint64(block.timestamp), 0, subjectRef);
            emit CredentialIssued(hashes[i], msg.sender, subjectRef);
        }
    }

    /// @notice Menjangkarkan akar Merkle sekelompok kredensial (satu transaksi untuk banyak kredensial).
    function anchorBatchRoot(bytes32 root, uint256 size) external onlyIssuer {
        if (batchRoots[root] != address(0)) revert AlreadyExists();
        batchRoots[root] = msg.sender;
        emit BatchAnchored(root, msg.sender, size);
    }

    function revoke(bytes32 credHash, string calldata reason) external {
        Record storage r = records[credHash];
        if (r.issuedAt == 0) revert UnknownCredential();
        if (r.issuer != msg.sender) revert NotOriginalIssuer();
        if (r.revokedAt != 0) revert AlreadyRevoked();
        r.revokedAt = uint64(block.timestamp);
        emit CredentialRevoked(credHash, msg.sender, reason);
    }

    function verify(bytes32 credHash) external view returns (
        bool exists, address issuer, bool issuerActive, uint64 issuedAt, bool revoked, uint64 revokedAt
    ) {
        Record memory r = records[credHash];
        exists = r.issuedAt != 0;
        issuer = r.issuer;
        issuerActive = issuers[r.issuer].active;
        issuedAt = r.issuedAt;
        revoked = r.revokedAt != 0;
        revokedAt = r.revokedAt;
    }

    /// @notice Verifikasi bukti keanggotaan Merkle (pasangan diurutkan) terhadap akar yang dijangkarkan.
    function verifyInBatch(bytes32 leaf, bytes32[] calldata proof, bytes32 root) external view returns (bool) {
        if (batchRoots[root] == address(0)) return false;
        bytes32 h = leaf;
        for (uint256 i = 0; i < proof.length; i++) {
            bytes32 p = proof[i];
            h = h < p ? keccak256(abi.encodePacked(h, p)) : keccak256(abi.encodePacked(p, h));
        }
        return h == root;
    }
}
'''
import solcx
solcx.install_solc('0.8.24')
out = solcx.compile_source(SOURCE, output_values=['abi','bin'], solc_version='0.8.24', optimize=True)
_, art = next((k,v) for k,v in out.items() if k.endswith(':CredentialRegistry'))
print('ukuran bytecode:', len(art['bin'])//2, 'byte')

ukuran bytecode: 3783 byte


## 2. Blockchain uji dalam memori dan deploy

In [3]:
from web3 import Web3
from web3.providers.eth_tester import EthereumTesterProvider
w3 = Web3(EthereumTesterProvider()); admin, issuer, lain = w3.eth.accounts[:3]
C = w3.eth.contract(abi=art['abi'], bytecode=art['bin'])
rc = w3.eth.wait_for_transaction_receipt(C.constructor().transact({'from': admin}))
reg = w3.eth.contract(address=rc.contractAddress, abi=art['abi'])
print('kontrak:', reg.address, '| gas deploy:', rc.gasUsed)
rc = w3.eth.wait_for_transaction_receipt(reg.functions.addIssuer(issuer, 'Lembaga Praktikum').transact({'from': admin}))
print('gas addIssuer:', rc.gasUsed)

kontrak: 0xF2E246BB76DF876Cef8b38ae84130F4F55De395b | gas deploy: 883669


gas addIssuer: 71915


## 3. Membuat kredensial VC 2.0 / Open Badges 3.0 dan menghitung hash

In [4]:
import json, uuid, datetime
from eth_utils import keccak
def kredensial(nama, capaian):
    return {'@context':['https://www.w3.org/ns/credentials/v2','https://purl.imsglobal.org/spec/ob/v3p0/context-3.0.3.json'],
            'id':f'urn:uuid:{uuid.uuid4()}','type':['VerifiableCredential','OpenBadgeCredential'],
            'issuer':{'id':f'did:pkh:eip155:{w3.eth.chain_id}:{issuer}','type':['Profile'],'name':'Lembaga Praktikum'},
            'validFrom':datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
            'credentialSubject':{'type':['AchievementSubject'],'name':nama,'achievement':{'type':['Achievement'],'name':capaian}}}
def h(c): return keccak(json.dumps(c, sort_keys=True, separators=(',',':')).encode())
k = kredensial('Peserta A', 'Dasar Smart Contract'); print(json.dumps(k, indent=1)[:400]); print('hash:', h(k).hex())

{
 "@context": [
  "https://www.w3.org/ns/credentials/v2",
  "https://purl.imsglobal.org/spec/ob/v3p0/context-3.0.3.json"
 ],
 "id": "urn:uuid:58c7c389-4075-41df-a213-efc2f6f33b1b",
 "type": [
  "VerifiableCredential",
  "OpenBadgeCredential"
 ],
 "issuer": {
  "id": "did:pkh:eip155:131277322940537:0x2B5AD5c4795c026514f8317c7a215E218DcCD6cF",
  "type": [
   "Profile"
  ],
  "name": "Lembaga Prakti
hash: 6cac4a64aa13054bc41f2508676ef6c222e8d16ae7f6404636bfc7136e4081a9


## 4. Terbitkan, verifikasi, rusak, dan cabut

In [5]:
gas = {}
rc = w3.eth.wait_for_transaction_receipt(reg.functions.issue(h(k), keccak(b'peserta-a')).transact({'from': issuer})); gas['issue'] = rc.gasUsed
print('verifikasi asli  :', reg.functions.verify(h(k)).call())
palsu = json.loads(json.dumps(k)); palsu['credentialSubject']['name'] = 'Peserta B'
print('verifikasi palsu :', reg.functions.verify(h(palsu)).call()[0])
rc = w3.eth.wait_for_transaction_receipt(reg.functions.revoke(h(k), 'uji').transact({'from': issuer})); gas['revoke'] = rc.gasUsed
print('setelah dicabut  :', reg.functions.verify(h(k)).call()); print('gas:', gas)

verifikasi asli  : [True, '0x2B5AD5c4795c026514f8317c7a215E218DcCD6cF', True, 1793464451, False, 0]
verifikasi palsu : False
setelah dicabut  : [True, '0x2B5AD5c4795c026514f8317c7a215E218DcCD6cF', True, 1793464451, True, 1793464452]
gas: {'issue': 73392, 'revoke': 49654}


## 5. Biaya gas: penerbitan per kredensial vs akar Merkle

In [6]:
def mroot(leaves):
    L = list(leaves)
    while len(L) > 1:
        L = [ (keccak(min(L[i],L[i+1])+max(L[i],L[i+1])) if i+1 < len(L) else L[i]) for i in range(0, len(L), 2)]
    return L[0]
import pandas as pd
rows = []
for n in [1, 5, 10, 20, 50]:
    hs = [keccak(f'b{n}-{i}'.encode()) for i in range(n)]
    g1 = w3.eth.wait_for_transaction_receipt(reg.functions.issueBatch(hs, b'\x00'*32).transact({'from': issuer})).gasUsed
    hs2 = [keccak(f'm{n}-{i}'.encode()) for i in range(n)]
    g2 = w3.eth.wait_for_transaction_receipt(reg.functions.anchorBatchRoot(mroot(hs2), n).transact({'from': issuer})).gasUsed
    rows.append({'n': n, 'gas/kred (issueBatch)': g1/n, 'gas/kred (Merkle)': g2/n})
df = pd.DataFrame(rows); df

,n,gas/kred (issueBatch),gas/kred (Merkle)
0,1,53934.0,48523.00
1,5,34543.8,9704.60
2,10,32120.1,4851.10
3,20,30906.8,2426.15
4,50,30181.2,970.46


## 6. Analitik pembelajaran dengan OULAD
Sumber: Kuzilek dkk. (2017), *Scientific Data* 4:170171, doi:10.1038/sdata.2017.171; salinan di UCI Machine Learning Repository (CC BY 4.0). Bila unduhan gagal, sel ini memakai **DATA SINTETIS** yang ditandai jelas.

In [7]:
import io, zipfile, urllib.request, numpy as np
URL = 'https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip'
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=120).read()))
    info = pd.read_csv(z.open('studentInfo.csv')); sa = pd.read_csv(z.open('studentAssessment.csv')); asm = pd.read_csv(z.open('assessments.csv'))
    sa = sa.merge(asm, on='id_assessment'); sa = sa[sa.date_submitted <= 60]; sa['score'] = pd.to_numeric(sa.score, errors='coerce')
    key = ['code_module','code_presentation','id_student']
    f = sa.groupby(key).agg(mean_score=('score','mean'), n_submit=('score','count')).reset_index()
    df = info.merge(f, on=key, how='left').fillna({'mean_score':0,'n_submit':0})
    df['y'] = df.final_result.isin(['Pass','Distinction']).astype(int); SUMBER = 'OULAD'
except Exception as e:
    print('unduhan gagal -> DATA SINTETIS:', e); r = np.random.default_rng(0)
    df = pd.DataFrame({'mean_score': r.uniform(0,100,3000), 'n_submit': r.integers(0,4,3000), 'num_of_prev_attempts': r.integers(0,3,3000), 'studied_credits': r.choice([30,60,120],3000)})
    df['y'] = (r.uniform(size=3000) < 1/(1+np.exp(-(0.04*df.mean_score+0.4*df.n_submit-3)))).astype(int); SUMBER = 'SINTETIS'
print(SUMBER, df.shape)

OULAD (32593, 15)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score
X = df[['mean_score','n_submit','num_of_prev_attempts','studied_credits']].values; y = df.y.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
m = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:,1]
print(f'[{SUMBER}] AUC = {roc_auc_score(yte,p):.3f} | F1 = {f1_score(yte,p>=.5):.3f}')

[OULAD] AUC = 0.855 | F1 = 0.776


## 7. Melampirkan bukti analitik ke kredensial
Probabilitas capaian dimasukkan sebagai `evidence`, lalu hash kredensial dijangkarkan.

In [9]:
k2 = kredensial('Peserta C', 'Analitik Data Pendidikan'); k2['evidence'] = [{'type':['Evidence'], 'name':'Bukti analitik', 'description': f'p(lulus)={p[0]:.2f}; model RF; data {SUMBER}'}]
rc = w3.eth.wait_for_transaction_receipt(reg.functions.issue(h(k2), keccak(b'peserta-c')).transact({'from': issuer}))
print('terbit dengan bukti, gas', rc.gasUsed, '| valid:', reg.functions.verify(h(k2)).call()[0])

terbit dengan bukti, gas 73380 | valid: True


## Latihan
1. Ubah kanonikalisasi menjadi JCS (RFC 8785) dan bandingkan hash.
2. Tambahkan fungsi `suspend` pada kontrak dan unit test-nya.
3. Bandingkan model dengan batas hari 30 vs 90.